# Waste Sorting Image Classifier

A four-class image classifier for household waste (PMD, Paper & Cardboard, Organic Waste, General Waste), trained on 400 AI-generated images. The notebook covers the dataset, three baselines (random guess, MLP, human), four CNN iterations from a small custom CNN to MobileNet transfer learning, and an error analysis.

## 1. Image dataset

### A: Dataset Creation

I created my image dataset using generative AI model instead of web scraping. The dataset includes four waste categories: PMD (Plastic, Metal, Drink Cartons), Paper & Cardboard, Organic Waste, and General Waste, with each class defined by detailed text prompts. I generated 100 images per prompt to ensure diversity in backgrounds, lighting, and angles. The images were then organized into labeled folders.

### B: Dataset Cleaning

I looked at each picture in my dataset one by one and removed any that looked unrealistic or were duplicates. I replaced them with better-quality images that fit the dataset. I also changed the prompts a little to add more variety and make sure the dataset had different types of images.

### C: Final Dataset

400 images (100 per class), generated with `src/generate_dataset.py`. The dataset isn't included in this repository; place it in `../data/dataset/<class name>/`.

### D: Load and Split

In [ ]:
dataset_path = "../data/dataset"  # one folder per class

## 2. Baselines

### A: Random Guess Accuracy

### **Random Guess Accuracy Calculation**

The **random guess accuracy** serves as a baseline reference, showing the accuracy we would expect if the model made predictions **completely at random**.

Since my dataset is **balanced**, with **4 classes** (PMD, Paper & Cardboard, Organic Waste, General Waste) having **equal representation**, we can calculate random guess accuracy using the formula:

$$
\text{Random Guess Accuracy} = \frac{100}{\text{Number of Classes}}\
$$

Substituting our values:

$$
\text{Random Guess Accuracy} = \frac{100}{4} = 25\%\
$$

### **Interpretation**
This means that if the model **randomly** predicts one of the four waste categories, it would be **correct 25% of the time**. 


### B: Basic Multilayer Perceptron Accuracy

#### I: Preprocessing

In [ ]:
import tensorflow as tf
import numpy as np
import os
import sklearn as ski
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow import keras

# Dataset path and class names
dataset_path = "../data/dataset"
classes_mlp = ["PMD", "Paper & Cardboard", "Organic Waste", "General Waste"]

# Load images and labels
images_mlp = []
labels_mlp = []

for class_name in classes_mlp:
    class_path = os.path.join(dataset_path, class_name)
    for filename in os.listdir(class_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(class_path, filename)
            # Load image with target size
            img = load_img(img_path, target_size=(128, 128))
            img_array = img_to_array(img) / 255.0  # Normalize the image
            img_array = img_array.flatten()  # Flatten the image for MLP input
            images_mlp.append(img_array)
            labels_mlp.append(class_name)

# Convert lists to NumPy arrays
images_mlp = np.array(images_mlp)
labels_mlp = np.array(labels_mlp)

# Label encoding using sklearn
label_encoder_mlp = ski.preprocessing.LabelEncoder()
y_integer_encoded_mlp = label_encoder_mlp.fit_transform(labels_mlp)

X_train_mlp, X_test_mlp, y_train_int_mlp, y_test_int_mlp = train_test_split(images_mlp, y_integer_encoded_mlp, test_size=0.1, random_state=42)
X_train_mlp, X_val_mlp, y_train_int_mlp, y_val_int_mlp = train_test_split(X_train_mlp, y_train_int_mlp, test_size=0.1, random_state=42)

# One-hot encode the labels
y_train_mlp = keras.utils.to_categorical(y_train_int_mlp, num_classes=len(classes_mlp))
y_val_mlp = keras.utils.to_categorical(y_val_int_mlp, num_classes=len(classes_mlp))
y_test_mlp = keras.utils.to_categorical(y_test_int_mlp, num_classes=len(classes_mlp))

print(f"Train set: {X_train_mlp.shape}, Validation set: {X_val_mlp.shape}, Test set: {X_test_mlp.shape}")

#### II: MLP Architecture 

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Define a function to create your MLP model
def mlp_model():
    return keras.Sequential([
        layers.Dense(512, activation="relu", input_shape=(128*128*3,)),  # Input layer
        layers.Dense(256, activation="relu"),  # Hidden layer
        layers.Dense(len(classes_mlp), activation="softmax")  # Output layer (4 classes)
    ])


#### III: MLP Training

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# Function to create the MLP model
def create_mlp_model():
    return keras.Sequential([
        layers.Dense(512, activation="relu", input_shape=(128*128*3,)),
        layers.Dense(256, activation="relu"),
        layers.Dense(len(classes_mlp), activation="softmax")
    ])

# Define early stopping callback
early_stopping_mlp = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# Define different learning rates to experiment with
learning_rates_mlp = [0.01, 0.001, 0.0001]
histories_mlp = {}

for lr in learning_rates_mlp:
    print(f"\nTraining with learning rate: {lr}\n")
    
    # Create a new instance of the model for the current learning rate
    model_mlp = create_mlp_model()
    
    # Compile the model with the current learning rate
    model_mlp.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                      loss="categorical_crossentropy",
                      metrics=["accuracy"])
    
    # Train the model
    history_mlp = model_mlp.fit(
        X_train_mlp, y_train_mlp,
        epochs=50,
        batch_size=32,
        validation_data=(X_val_mlp, y_val_mlp),
        callbacks=[early_stopping_mlp],
        verbose=1
    )
    
    # Store training history
    histories_mlp[lr] = history_mlp
    
    # Get the final training and validation accuracies
    final_train_acc = history_mlp.history["accuracy"][-1]
    final_val_acc = history_mlp.history["val_accuracy"][-1]
    
    # Evaluate the model on the test set
    test_loss, test_acc = model_mlp.evaluate(X_test_mlp, y_test_mlp, verbose=0)
    
    print(f"Learning rate {lr}:")
    print(f"  Test accuracy: {test_acc:.4f}")
    print(f"  Test loss: {test_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot accuracy and loss for each learning rate
for lr in histories_mlp.keys():
    history_mlp = histories_mlp[lr]  # Get the history object for the current learning rate

    print(f"\nLearning Curves for Learning Rate: {lr}\n")

    # Create a new figure for each learning rate
    plt.figure(figsize=(12, 5))

    # Plot Training vs Validation Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history_mlp.history["accuracy"], label="Train Accuracy")
    plt.plot(history_mlp.history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.title(f"Training vs Validation Accuracy (LR={lr})")

    # Plot Training vs Validation Loss
    plt.subplot(1, 2, 2)
    plt.plot(history_mlp.history["loss"], label="Train Loss")
    plt.plot(history_mlp.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.title(f"Training vs Validation Loss (LR={lr})")

    # Show the plots
    plt.show()

#### IV: MLP Evaluation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Predict probabilities on the test set
y_pred_probs = model_mlp.predict(X_test_mlp)

# Convert probabilities to predicted class labels
y_pred = np.argmax(y_pred_probs, axis=1)

# Convert one-hot encoded test labels to class indices
y_true = np.argmax(y_test_mlp, axis=1)

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

# Generate and print the classification report
report = classification_report(y_true, y_pred)
print("Classification Report:")
print(report)

# Calculate and print overall test accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

## **Final MLP Model Evaluation**

The best learning rate for the MLP model was **0.001**, selected based on its **highest test accuracy** and consistent performance across training, validation, and test sets.

### **Why is 0.001 the Best Learning Rate?**
- **Highest Test Accuracy** – It achieved a test accuracy of **67.50%**, outperforming the other learning rates (62.50% for 0.01 and 65.00% for 0.0001).
- **Stable Loss** – Although its test loss (1.0631) is comparable to or slightly higher than the others, the overall performance indicates good convergence.
- **Best Generalization** – The model trained with a 0.001 learning rate shows a good balance between fitting the training data and generalizing to unseen data, avoiding both underfitting and overfitting.
- **Compared to Others** – The other learning rates yielded lower accuracies and/or less consistent performance.

### **Final Test Performance**
$$
\text{Final Test Accuracy} = 67.50\%
$$

$$
\text{Final Test Loss} = 1.0631
$$

### C: Human-Level Performance Accuracy

### Human-Level Performance Accuracy Calculation

To calculate the human-level performance accuracy, I created a Microsoft Forms quiz consisting of **40 multiple-choice questions**. Each question displayed one image from my dataset, and participants were asked to classify it as one of the following four categories:

- PMD  
- General Waste  
- Organic Waste  
- Paper & Cardboard

The quiz included **10 images per class**, ensuring a balanced representation across all categories.

I collected responses from **three individuals**: a peer, a friend, and a family member. This allowed me to gather perspectives from a small but diverse group to reduce potential bias.

According to Microsoft Forms, the **average score** across all participants was:

- **31.3 out of 40 questions correct**

This gives us the human-level accuracy using the formula:

**Human-Level Accuracy** = $\left( \frac{31.3}{40} \right) \times 100 = \mathbf{78.25\%}$

This average reflects the baseline accuracy that a non-expert but attentive human might achieve on this classification task. It provides a useful benchmark to compare against the performance of our trained image classifier.

## 3. Iteration 1: basic CNN

### A: Preprocessing

### Preprocessing Steps Explanation

Starting off by loading the data, I resized all images to a consistent shape of 128x128 pixels. I then normalized the pixel values by dividing them by 255, scaling them to a range of 0 to 1 for better training performance. After that, I converted both the images and labels into NumPy arrays for efficient processing. The class labels were encoded from strings to integers using a label encoder, and then one-hot encoded to make them suitable for multi-class classification. I also saved the order of class names after encoding to prevent any issues related to automatic alphabetical sorting. Finally, I split the data into training, validation, and test sets to prepare for model training and evaluation.

In [ ]:
import tensorflow as tf
import numpy as np
import os
import sklearn as ski
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow import keras
import random 

# Dataset path and class names
dataset_path = "../data/dataset"
classes_it1 = ["General Waste", "Organic Waste", "Paper & Cardboard", "PMD"]

# Load images and labels
images_it1 = []
labels_it1 = []

for class_name in classes_it1:
    class_path = os.path.join(dataset_path, class_name)
    for filename in os.listdir(class_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(class_path, filename)
            # Load image with target size
            img = load_img(img_path, target_size=(128, 128))
            img_array = img_to_array(img) / 255.0  # Normalize the image
            images_it1.append(img_array)
            labels_it1.append(class_name)

# Convert lists to NumPy arrays
images_it1 = np.array(images_it1)
labels_it1 = np.array(labels_it1)

# Label encoding using sklearn
label_encoder_it1 = ski.preprocessing.LabelEncoder()
y_integer_encoded_it1 = label_encoder_it1.fit_transform(labels_it1)

# One-hot encode the labels
y_onehot_encoded_it1 = keras.utils.to_categorical(y_integer_encoded_it1, num_classes=len(classes_it1))

# After fitting the encoder
classes_it1 = list(label_encoder_it1.classes_)

# Split dataset: 80% Train, 10% Validation, 10% Test
X_train_it1, X_test_it1, y_train_it1, y_test_it1 = train_test_split(images_it1, y_onehot_encoded_it1, test_size=0.1, random_state=42)
X_train_it1, X_val_it1, y_train_it1, y_val_it1 = train_test_split(X_train_it1, y_train_it1, test_size=0.1, random_state=42)

print(f"Train set: {X_train_it1.shape}, Validation set: {X_val_it1.shape}, Test set: {X_test_it1.shape}")

### B: CNN Architecture

### Strategy to Train the CNN Model
I chose three convolutional blocks as a balanced approach to progressively extract low-, mid-, and high-level features from the images. Initially, I tried using an increasing number of filters per layer, but the model started to overfit. So instead, I switched to a descending order of filters (128 > 64 > 32), which worked better for my data. Each block uses ReLU activation, max pooling, with batch normalization after the first layer to speed up training and some dropout to prevent overfitting. I originally used `Flatten()` to transition to the dense layers, but replacing it with GlobalAveragePooling2D improved generalization and helped reduce overfitting. The final layers include a ReLU-activated dense layer followed by a softmax output layer for multi-class classification.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, GlobalAveragePooling2D, Dense

def build_model(num_classes):
    model = Sequential([
        # First Convolutional Block
        Conv2D(128, (3, 3), padding='same', activation='relu', input_shape=(128, 128, 3)),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        #Dropout(0.15),  
        
        # Second Convolutional Block
        Conv2D(64, (3, 3), padding='same', activation='relu'),
        #BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        #Dropout(0.15),
        
        # Third Convolutional Block
        Conv2D(32, (3, 3), padding='same', activation='relu'),
        #BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.15),
        
        # Transition to Fully Connected Layers
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        #BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model

model_it1 = build_model(num_classes=len(classes_it1))
model_it1.summary()

### C: CNN Training

### Training Strategy Explanation

I compiled the model using the Adam optimizer with a learning rate of `0.0001` to compare its performance with my MLP model. I used categorical cross-entropy as the loss function for this multi-class classification task and included accuracy as a metric. To reduce overfitting, I applied EarlyStopping with a patience of 3 and `restore_best_weights=True`. The model was trained on both training and validation data for 50 epochs with a batch size of 32. After training, I plotted the loss and accuracy curves to observe the learning behavior. Finally, I evaluated the model on the test set to get an unbiased performance result.

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping

# Define EarlyStopping callback with updated patience
early_stop_it1 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Compile the updated model
model_it1.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

# Train the updated model with training and validation data
history_it1 = model_it1.fit(X_train_it1, y_train_it1,
                            validation_data=(X_val_it1, y_val_it1),
                            epochs=50,
                            batch_size=32,
                            callbacks=[early_stop_it1])

# Plot learning curves for loss and accuracy
plt.figure(figsize=(12, 5))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(history_it1.history['loss'], marker='o', label='Training Loss')
plt.plot(history_it1.history['val_loss'], marker='o', label='Validation Loss')
plt.title('Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(history_it1.history['accuracy'], marker='o', label='Training Accuracy')
plt.plot(history_it1.history['val_accuracy'], marker='o', label='Validation Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# Evaluate the updated model on the test set
test_loss_it1, test_acc_it1 = model_it1.evaluate(X_test_it1, y_test_it1)
print(f"Test Loss: {test_loss_it1:.4f}, Test Accuracy: {test_acc_it1:.4f}")

### D: Model Evaluation

### **Model Evaluation Process**

I evaluated the model using the test set by predicting class probabilities and converting them to class labels. I then compared the predictions with the true labels using a confusion matrix and a classification report to assess performance across all classes. To get a comprehensive view, I calculated the overall accuracy, as well as the weighted precision, recall, and F1-score, which account for class imbalance. This helped me understand not just how often the model was correct, but how well it performed across all categories.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# Make predictions on the test set
y_pred_it1 = model_it1.predict(X_test_it1)

# Convert predicted probabilities to predicted class indices
y_pred_class_it1 = np.argmax(y_pred_it1, axis=1)

# Convert one-hot encoded true labels to class indices
y_true_it1 = np.argmax(y_test_it1, axis=1)

# Compute the confusion matrix
cm_it1 = confusion_matrix(y_true_it1, y_pred_class_it1)
print("Confusion Matrix:")
print(cm_it1)

# Generate a classification report
report_it1 = classification_report(y_true_it1, y_pred_class_it1, target_names=classes_it1)
print("Classification Report:")
print(report_it1)

# Compute overall metrics
accuracy_it1 = accuracy_score(y_true_it1, y_pred_class_it1)
precision_it1 = precision_score(y_true_it1, y_pred_class_it1, average="weighted")
recall_it1 = recall_score(y_true_it1, y_pred_class_it1, average="weighted")
f1_it1 = f1_score(y_true_it1, y_pred_class_it1, average="weighted")

print(f"Accuracy: {accuracy_it1:.4f}")
print(f"Precision (weighted): {precision_it1:.4f}")
print(f"Recall (weighted): {recall_it1:.4f}")
print(f"F1 Score (weighted): {f1_it1:.4f}")

### Evaluation of the First CNN Model

The model reached 72.5% accuracy with a weighted F1-score of 71.2%. It performed best on Organic Waste, with perfect precision and recall. General Waste and PMD had moderate performance, while Paper & Cardboard showed high precision but low recall, indicating missed predictions. The confusion matrix highlights some misclassification, especially between visually similar classes. Overall, the model shows good generalization but could improve on class balance and recall.

## 4. Iteration 2: data augmentation

### A: Preprocessing

### Preprocessing Steps for Data Augmentation

In this iteration, I defined a custom class order to resolve a recurring issue where labels were incorrectly mapped during training and evaluation. This was caused by LabelEncoder's default alphabetical sorting, which didn’t match the folder structure. By manually specifying the class order and fitting the encoder with it, I ensured consistent and correct label-to-index mapping.

I then resized all images to 128×128, normalized pixel values to [0–1], and encoded labels using the pre-fitted encoder. The data was converted to NumPy arrays, one-hot encoded for classification, and split into 80% training, 10% validation, and 10% test sets

To improve generalization and reduce overfitting, I applied data augmentation to the training set using `ImageDataGenerator`, including random rotations, shifts, zoom, shearing, and horizontal flips. These transformations help the model learn more robust features by simulating real-world variations. I initially added brightness augmentation to reflect varying lighting conditions, but it caused some images to appear completely black, so I removed it. For validation and test sets, no augmentation was applied to keep evaluation consistent. I also visualized a batch of augmented images to verify the transformations.


In [ ]:
import tensorflow as tf
import numpy as np
import os
import sklearn as ski
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder

# Dataset path and class names
dataset_path_it2 = "../data/dataset"
classes_it2 = ["PMD", "Paper & Cardboard", "Organic Waste", "General Waste"]

# Create and fit LabelEncoder my custom class order
label_encoder_it2 = LabelEncoder()
label_encoder_it2.fit(classes_it2)

# Load images and labels
images_it2 = []
labels_encoded_it2 = []  

for class_name in classes_it2:
    class_path = os.path.join(dataset_path_it2, class_name)
    for filename in os.listdir(class_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(class_path, filename)
            # Load image with target size and normalize to [0, 1]
            img = load_img(img_path, target_size=(128, 128))
            img_array = img_to_array(img) / 255.0
            images_it2.append(img_array)
            # Transform the label using the pre-fitted label encoder
            label_index = label_encoder_it2.transform([class_name])[0]
            labels_encoded_it2.append(label_index)

# Convert lists to NumPy arrays
images_it2 = np.array(images_it2)
labels_encoded_it2 = np.array(labels_encoded_it2)

# One-hot encode the labels using the number of classes from your custom list
y_onehot_encoded_it2 = keras.utils.to_categorical(labels_encoded_it2, num_classes=len(classes_it2))

# After fitting the encoder
classes_it2 = list(label_encoder_it2.classes_)

# Split dataset: 80% Train, 10% Validation, 10% Test for iteration 2
X_train_it2, X_test_it2, y_train_it2, y_test_it2 = train_test_split(images_it2, y_onehot_encoded_it2, test_size=0.1, random_state=42)
X_train_it2, X_val_it2, y_train_it2, y_val_it2 = train_test_split(X_train_it2, y_train_it2, test_size=0.1, random_state=42)

print(f"Train set: {X_train_it2.shape}, Validation set: {X_val_it2.shape}, Test set: {X_test_it2.shape}")


In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen_it2 = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,     
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    horizontal_flip=True,
    #brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# For validation and test sets, use generators without augmentation (to preserve evaluation integrity)
val_datagen_it2 = ImageDataGenerator()
test_datagen_it2 = ImageDataGenerator()

# Create a training data generator for iteration 2 using the iteration-specific training data
train_generator_it2 = train_datagen_it2.flow(X_train_it2, y_train_it2, batch_size=32)

# Visualize a sample of augmented images for iteration 2
sample_images_it2, _ = next(train_generator_it2)
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(sample_images_it2[i], vmin=0, vmax=1)  # Ensure proper brightness scaling
    plt.axis('off')
plt.suptitle("Iteration 2: Sample Augmented Training Images")
plt.tight_layout()
plt.show()

### B: CNN Architecture

For the architecture, I reused the same model architecture from the first iteration by calling the previously defined `build_model()` function. I did this to ensure a fair comparison between the models from different iterations by keeping the network structure consistent.

In [ ]:
model_it2 = build_model(num_classes=len(classes_it2))
model_it2.summary()

### C: CNN Training

For training, I compiled the model using the Adam optimizer with a learning rate of [0.0001], along with categorical cross-entropy loss and accuracy as the evaluation metric, matching the settings used in the first iteration for consistency. I used an EarlyStopping callback to monitor validation loss and stop training if it didn’t improve for 3 consecutive epochs, while also restoring the best weights.

The model was trained on the augmented training data using a generator, which feeds batches of transformed images during training. I trained it for up to 50 epochs with the same batch size and step configuration as in the first iteration. To visualize the learning process, I plotted the training and validation loss and accuracy curves, which helped me monitor the model's performance and check for signs of overfitting or underfitting.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# Define EarlyStopping callback for iteration 2 (matching first iteration settings)
early_stop_it2 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Compile the CNN model for iteration 2
model_it2.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

# Train the model using the augmented training generator for iteration 2
history_it2 = model_it2.fit(
    train_generator_it2,
    steps_per_epoch=len(X_train_it2) // 32,
    epochs=50,
    validation_data=(X_val_it2, y_val_it2),
    callbacks=[early_stop_it2]
)

# Plot learning curves for loss and accuracy for iteration 2
plt.figure(figsize=(12, 5))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(history_it2.history['loss'], marker='o', label='Training Loss')
plt.plot(history_it2.history['val_loss'], marker='o', label='Validation Loss')
plt.title('Iteration 2: Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(history_it2.history['accuracy'], marker='o', label='Training Accuracy')
plt.plot(history_it2.history['val_accuracy'], marker='o', label='Validation Accuracy')
plt.title('Iteration 2: Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# Save the trained model
os.makedirs("../models", exist_ok=True)
model_it2.save("../models/model_it2.h5")

### D: Model Evaluation

For evaluation, I first tested the model on the test set to obtain the final loss and accuracy, giving an overall view of the model’s performance on unseen data. I then generated predictions and used them to compute the confusion matrix, which shows how well the model distinguished between classes. To gain deeper insight, I printed a classification report including precision, recall, and F1-score for each class.

In addition to class-wise metrics, I calculated the overall weighted accuracy, precision, recall, and F1-score, which account for class imbalance. This comprehensive evaluation allowed me to assess both the general performance and how well the model handled each individual waste category.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# Evaluate the model on the iteration 2 test set
test_loss_it2, test_acc_it2 = model_it2.evaluate(X_test_it2, y_test_it2, verbose=0)
print(f"Iteration 2 Test Loss: {test_loss_it2:.4f}, Test Accuracy: {test_acc_it2:.4f}")

# Generate predictions for the test set
y_pred_it2 = model_it2.predict(X_test_it2)
y_pred_class_it2 = np.argmax(y_pred_it2, axis=1)
y_true_it2 = np.argmax(y_test_it2, axis=1)

# Compute the confusion matrix
cm_it2 = confusion_matrix(y_true_it2, y_pred_class_it2)
print("Confusion Matrix:")
print(cm_it2)

# Print the classification report
report_it2 = classification_report(y_true_it2, y_pred_class_it2, target_names=classes_it2)
print("Classification Report:")
print(report_it2)

# Compute and print overall metrics
accuracy_it2 = accuracy_score(y_true_it2, y_pred_class_it2)
precision_it2 = precision_score(y_true_it2, y_pred_class_it2, average='weighted')
recall_it2 = recall_score(y_true_it2, y_pred_class_it2, average='weighted')
f1_it2 = f1_score(y_true_it2, y_pred_class_it2, average='weighted')

print(f"Accuracy: {accuracy_it2:.4f}")
print(f"Precision (weighted): {precision_it2:.4f}")
print(f"Recall (weighted): {recall_it2:.4f}")
print(f"F1 Score (weighted): {f1_it2:.4f}")


### Evaluation of Iteration 2 CNN Model

The model achieved a test accuracy of 85.0% with a test loss of 0.5676, indicating strong performance on unseen data. The weighted precision (87.0%), recall (85.0%), and F1-score (84.7%) indicate that data augmentation helped improve the model’s robustness and generalization. While results are solid, the remaining misclassifications highlight areas for further tuning.

## 5. Iteration 3: transfer learning (MobileNet)

### A: Preprocessing

In this iteration, I used a pre-trained MobileNet model, which required a different preprocessing approach. I resized all images to 224×224 pixels, matching MobileNet’s expected input size. Unlike previous iterations, I skipped manual normalization and instead applied MobileNet’s `preprocess_input()`, which scales pixel values to the [-1, 1] range.

I loaded the images and their labels, then used LabelEncoder to convert the class names into integers, followed by one-hot encoding for multi-class classification. After converting the data into NumPy arrays, I split the dataset into 80% training, 10% validation, and 10% testing.

In [ ]:
import tensorflow as tf
import numpy as np
import os
import sklearn as ski
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow import keras

# Dataset path and class names
dataset_path = "../data/dataset"
classes_it3 = ["PMD", "Paper & Cardboard", "Organic Waste", "General Waste"]

# Load images and labels
images = []
labels = []
for class_name in classes:
    class_path = os.path.join(dataset_path, class_name)
    for filename in os.listdir(class_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(class_path, filename)
            img = load_img(img_path, target_size=(224, 224))
            img_array = img_to_array(img)  # No normalize here; MobileNet's preprocess_input will handle it.
            images.append(img_array)
            labels.append(class_name)

# Convert lists to NumPy arrays
images = np.array(images)
labels = np.array(labels)

# Preprocess images using MobileNet's preprocess_input (scales pixels to [-1, 1])
preprocessed_images = tf.keras.applications.mobilenet.preprocess_input(images)

# Label encoding using sklearn and one-hot encoding
label_encoder_it3 = ski.preprocessing.LabelEncoder()
y_integer_encoded = label_encoder_it3.fit_transform(labels)
y_onehot_encoded = keras.utils.to_categorical(y_integer_encoded, num_classes=len(classes_it3))

# After fitting the encoder
classes_it3 = list(label_encoder_it3.classes_)

# Split dataset: 80% Train, 10% Validation, 10% Test
X_train_it3, X_test_it3, y_train_it3, y_test_it3 = train_test_split(preprocessed_images, y_onehot_encoded, test_size=0.1, random_state=42)
X_train_it3, X_val_it3, y_train_it3, y_val_it3 = train_test_split(X_train_it3, y_train_it3, test_size=0.1, random_state=42)

print(f"Train set: {X_train_it3.shape}, Validation set: {X_val_it3.shape}, Test set: {X_test_it3.shape}")

### B: CNN Architecture

For the architecture, I chose MobileNet as a pre-trained base model to extract high-level features from my image data, and for its efficiency and ability to generalize well on small datasets, making it ideal for my waste classification task. By setting `include_top=False`, I removed the original classification head and used only the convolutional layers. These layers were frozen (trainable=False) to retain the learned ImageNet features while training only the custom classifier on top.

On top of MobileNet, I built a custom dense classifier consisting of a GlobalAveragePooling2D layer to reduce dimensionality, followed by two dense layers (128 and 256 units with ReLU activation), a Dropout layer for regularization, and a final softmax layer for multi-class classification.

In [ ]:
from tensorflow.keras import layers, models, applications

# Define the MobileNet base model for feature extraction
base_model = applications.MobileNet(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_model.trainable = False

# Build the dense classifier using the provided layer format
dense_classifier = models.Sequential([
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dropout(rate=0.3),
    layers.Dense(len(classes_it3), activation='softmax')
])

# Combine the MobileNet base and the dense classifier
model_it3 = models.Sequential([base_model, dense_classifier])
model_it3.summary()


### C: CNN Training

For training, I compiled the transfer learning model using the Adam optimizer with a learning rate of 0.0001, along with categorical cross-entropy loss and accuracy as the evaluation metric. To prevent overfitting, I used EarlyStopping to monitor validation loss and stop training if it didn’t improve after 3 epochs, while restoring the best weights.

The model was trained for up to 50 epochs with a batch size of 32, using the preprocessed training and validation sets. After training, I evaluated the model on the test set to assess its generalization performance, reporting both test loss and accuracy as final performance indicators.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# Compile the transfer learning model using MobileNet as the base
model_it3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Set up EarlyStopping to monitor validation loss and prevent overfitting
early_stop_it3 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model (assumes X_train_it3, y_train_it3, X_val_it3, y_val_it3 are defined)
history_it3 = model_it3.fit(
    X_train_it3, y_train_it3,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_it3, y_val_it3),
    callbacks=[early_stop_it3],
    verbose=1
)

# Evaluate the model on the test set (assumes X_test_it3 and y_test_it3 are defined)
test_loss_it3, test_acc_it3 = model_it3.evaluate(X_test_it3, y_test_it3, verbose=0)
print(f"Iteration 3 Test Loss: {test_loss_it3:.4f}, Test Accuracy: {test_acc_it3:.4f}")


### D: Model Evaluation

After training, I evaluated the model on the test set to measure its performance on unseen data, reporting the final test loss and accuracy. I then generated predictions and used them to compute a confusion matrix and a detailed classification report, which provided class-wise precision, recall, and F1-scores.

To better understand the model's learning behavior, I also plotted the training and validation loss and accuracy curves over the epochs. These plots helped visualize the model’s convergence and detect any signs of overfitting or underfitting during training. This comprehensive evaluation provided insights into both the overall and class-specific performance of the MobileNet-based model.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# Evaluate the CNN model (Iteration 3) on the test set
test_loss_it3, test_acc_it3 = model_it3.evaluate(X_test_it3, y_test_it3, verbose=0)
print(f"Iteration 3 Test Loss: {test_loss_it3:.4f}, Test Accuracy: {test_acc_it3:.4f}")

# Generate predictions for the test set
y_pred_it3 = model_it3.predict(X_test_it3)
y_pred_class_it3 = np.argmax(y_pred_it3, axis=1)
y_true_it3 = np.argmax(y_test_it3, axis=1)

# Compute the confusion matrix
cm_it3 = confusion_matrix(y_true_it3, y_pred_class_it3)
print("Confusion Matrix:")
print(cm_it3)

# Print the classification report
report_it3 = classification_report(y_true_it3, y_pred_class_it3, target_names=classes_it3)
print("Classification Report:")
print(report_it3)

# Plot learning curves for training and validation loss/accuracy
plt.figure(figsize=(12, 5))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(history_it3.history['loss'], marker='o', label='Training Loss')
plt.plot(history_it3.history['val_loss'], marker='o', label='Validation Loss')
plt.title('Iteration 3: Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(history_it3.history['accuracy'], marker='o', label='Training Accuracy')
plt.plot(history_it3.history['val_accuracy'], marker='o', label='Validation Accuracy')
plt.title('Iteration 3: Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


### Evaluation of iteration 3 Pre-trained model

The MobileNet-based model achieved a test accuracy of 97.5% and a low test loss of 0.0382, demonstrating excellent generalization. The confusion matrix showed near-perfect classification, with only one misclassified instance in the PMD class. All other classes were predicted with 100% recall.

The classification report confirms consistently high scores across all categories, with precision, recall, and F1-scores close to or at 1.00. The learning curves show smooth convergence and no overfitting, with training and validation accuracy staying high throughout.

These results confirm that transfer learning with MobileNet was highly effective, clearly outperforming previous iterations in both overall and class-wise performance.

## 6. Iteration 4: MobileNet with built-in augmentation

### A: Preprocessing

In this iteration, I continued using MobileNet, so I resized all images to 224×224 pixels, which is the expected input size for the model. I used MobileNet’s `preprocess_input() ` to scale pixel values to the [-1, 1] range for optimal performance with pre-trained weights.

The class labels were encoded using `LabelEncoder` and then one-hot encoded for multi-class classification. After converting the images and labels to NumPy arrays, I split the dataset into 80% training, 10% validation, and 10% testing, maintaining the same data structure as in previous iterations to ensure fair comparison.

In [ ]:
import tensorflow as tf
import numpy as np
import os
import sklearn as ski
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow import keras

# Dataset path and class names for iteration 4
dataset_path_it4 = "../data/dataset"
classes_it4 = ["PMD", "Paper & Cardboard", "Organic Waste", "General Waste"]

# Load images and labels for iteration 4
images_it4 = []
labels_it4 = []
for class_name in classes_it4:
    class_path = os.path.join(dataset_path_it4, class_name)
    for filename in os.listdir(class_path):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            img_path = os.path.join(class_path, filename)
            # Load image with target size (MobileNet accepts 128x128 images)
            img = load_img(img_path, target_size=(224, 224))
            img_array = img_to_array(img)  # Raw pixel values
            images_it4.append(img_array)
            labels_it4.append(class_name)
images_it4 = np.array(images_it4)
labels_it4 = np.array(labels_it4)

# Preprocess images using MobileNet's preprocess_input (scales pixels to [-1, 1])
images_it4 = tf.keras.applications.mobilenet.preprocess_input(images_it4)

# Label encoding and one-hot encoding
label_encoder_it4 = ski.preprocessing.LabelEncoder()
y_integer_encoded_it4 = label_encoder_it4.fit_transform(labels_it4)
y_onehot_encoded_it4 = keras.utils.to_categorical(y_integer_encoded_it4, num_classes=len(classes_it4))

# After fitting the encoder
classes_it4 = list(label_encoder_it4.classes_)

# Split dataset: 80% Train, 10% Validation, 10% Test for iteration 4
X_train_it4, X_test_it4, y_train_it4, y_test_it4 = train_test_split(
    images_it4, y_onehot_encoded_it4, test_size=0.1, random_state=42)
X_train_it4, X_val_it4, y_train_it4, y_val_it4 = train_test_split(
    X_train_it4, y_train_it4, test_size=0.1, random_state=42)

print(f"Train set: {X_train_it4.shape}, Validation set: {X_val_it4.shape}, Test set: {X_test_it4.shape}")

### B: CNN Architecture

For this iteration, I used MobileNet as a frozen base model for transfer learning, but introduced an integrated data augmentation pipeline directly into the model architecture. This included random flips, rotations, zooms, translations, and brightness adjustments to improve the model’s ability to generalize to real-world variability.

The input passes through this augmentation layer before reaching the MobileNet base, which is set to non-trainable to retain its pre-trained ImageNet features. On top of the base, I added a custom classifier with GlobalAveragePooling, two dense layers (128 and 256 units), a Dropout layer (0.2) for regularization, and a final softmax output layer for multi-class classification.

In [ ]:
from tensorflow.keras import Input, Model
from tensorflow.keras import layers, models, applications

# Define your data augmentation Sequential model
data_augmentation_it4 = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),           # ~15 degrees
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.Lambda(lambda x: tf.image.random_brightness(x, max_delta=0.2))
])

# Input layer
input_tensor = Input(shape=(224, 224, 3))

# Apply data augmentation
x = data_augmentation_it4(input_tensor)

# Load MobileNet base model (with input_tensor disabled now, since we're using `x`)
base_model_it4 = applications.MobileNet(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_model_it4.trainable = False

# Pass the augmented input through the base model
x = base_model_it4(x, training=False)

# Add your custom dense classifier
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.2)(x)
output_tensor = layers.Dense(len(classes_it4), activation='softmax')(x)

# Build the complete model
model_it4 = Model(inputs=input_tensor, outputs=output_tensor)

# Summary
model_it4.summary()


### C: CNN Training

The model was compiled using the Adam optimizer with a learning rate of 0.0001 and categorical cross-entropy loss for multi-class classification. To prevent overfitting, EarlyStopping was used to monitor validation loss, restoring the best weights after 3 stagnant epochs.

Training was run for up to 50 epochs with a batch size of 32, using augmented training data and a validation split. After training, the model was evaluated on the test set to assess final performance, reporting both test loss and accuracy.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# Compile the model with Adam optimizer and categorical crossentropy loss
model_it4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Set up EarlyStopping to monitor validation loss and prevent overfitting
early_stop_it4 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model on the training set with validation data
history_it4 = model_it4.fit(
    X_train_it4, y_train_it4,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_it4, y_val_it4),
    callbacks=[early_stop_it4],
    verbose=1
)

test_loss_it4, test_acc_it4 = model_it4.evaluate(X_test_it4, y_test_it4, verbose=0)
print(f"Iteration 4 Test Loss: {test_loss_it4:.4f}, Test Accuracy: {test_acc_it4:.4f}")

### D: Model Evaluation

The model was evaluated on the test set, reporting final loss and accuracy. Predictions were used to generate a confusion matrix and classification report, showing precision, recall, and F1-score per class. The results helped assess class-wise performance and identify any misclassifications. Additionally, training and validation curves were plotted to visualize learning behavior and confirm model stability.

In [ ]:
# Evaluate the model on the test set
test_loss_it4, test_acc_it4 = model_it4.evaluate(X_test_it4, y_test_it4, verbose=0)
print(f"Iteration 4 Test Loss: {test_loss_it4:.4f}, Test Accuracy: {test_acc_it4:.4f}")

# Generate predictions for the test set and compute evaluation metrics
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_pred_it4 = model_it4.predict(X_test_it4)
y_pred_class_it4 = np.argmax(y_pred_it4, axis=1)
y_true_it4 = np.argmax(y_test_it4, axis=1)

# Compute and display the confusion matrix
cm_it4 = confusion_matrix(y_true_it4, y_pred_class_it4)
print("Confusion Matrix:")
print(cm_it4)

# Print the classification report
report_it4 = classification_report(y_true_it4, y_pred_class_it4, target_names=classes_it4)
print("Classification Report:")
print(report_it4)

# Plot learning curves for training and validation loss and accuracy
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_it4.history['loss'], marker='o', label='Training Loss')
plt.plot(history_it4.history['val_loss'], marker='o', label='Validation Loss')
plt.title('Iteration 4: Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_it4.history['accuracy'], marker='o', label='Training Accuracy')
plt.plot(history_it4.history['val_accuracy'], marker='o', label='Validation Accuracy')
plt.title('Iteration 4: Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

### Evaluation on iteration 4 Pre-trained model with augmentation

The model achieved a perfect test accuracy of 100% and an extremely low test loss of 0.0133, indicating excellent generalization and flawless performance on the test set. The confusion matrix confirms that all 40 test samples were correctly classified across all four classes, with no misclassifications.

The classification report shows perfect scores (1.00) for precision, recall, and F1-score in every class, including PMD, Paper & Cardboard, Organic Waste, and General Waste. The learning curves show smooth and consistent convergence, with training and validation loss steadily decreasing and accuracy reaching 100% early in training.

These results demonstrate that integrating data augmentation directly into the model, combined with MobileNet’s transfer learning, led to highly robust feature learning and exceptional classification performance.

## 7. Error analysis

### A: Wrong Classifications

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get predictions on the test set for iteration 2
predictions_it2 = model_it2.predict(X_test_it2)

# The output of the model and the y labels are arrays of 2 elements (one per class)
# Use argmax to get the index of the maximum value in the array.
predicted_labels_it2 = np.argmax(predictions_it2, axis=1)
actual_labels_it2 = np.argmax(y_test_it2, axis=1)

# Find instances where predictions do not match labels
mismatches_it2 = np.where(predicted_labels_it2 != actual_labels_it2)[0]

# Store the misclassified images for error analysis
wrong_predictions_it2 = X_test_it2[mismatches_it2]

# Now you can analyze 'wrong_predictions_it2' for error analysis
# Display a sample of misclassified images (up to 10)
num_to_display = min(10, len(mismatches_it2))
plt.figure(figsize=(15, 10))
for i in range(num_to_display):
    idx = mismatches_it2[i]
    # Reverse MobileNet preprocessing: scale pixel values from [-1, 1] back to [0, 1]
    image = (X_test_it2[idx] + 1) / 2.0
    plt.subplot(2, 5, i + 1)
    plt.imshow(image.astype('float32'))
    plt.title(f"True: {classes_it2[actual_labels_it2[idx]]}\nPred: {classes_it2[predicted_labels_it2[idx]]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

### B: Identify Error Types

| Image | True Label       | Predicted Label    | Error Type               | Possible Reason                                                                                           |
|-------|------------------|--------------------|--------------------------|-----------------------------------------------------------------------------------------------------------|
| 1     | PMD              | General Waste      | Material Confusion       | Items like cans and cartons have neutral tones that can resemble general waste.                           |
| 2     | PMD              | General Waste      | Visual Clutter           | Mixed packaging and reflective surfaces create a cluttered scene that confuses the model.                  |
| 3     | PMD              | General Waste      | Context-based Confusion  | A food court or cafeteria-like setting may bias the prediction toward general waste.                       |
| 4     | General Waste    | Paper & Cardboard  | Dominant Shape/Color Bias| Scattered white items and textures can mimic paper/cardboard characteristics.                              |
| 5     | Paper & Cardboard| PMD                | Shape and Texture Overlap| Folded or compacted cardboard may resemble plastic or foil packaging typical of PMD.                       |
| 6     | General Waste    | Organic Waste      | Texture and Color Similarity | Soft textures and natural tones can be mistaken for organic materials.                               |
| 7     | General Waste    | PMD                | Object-level Ambiguity   | Crumpled paper or small items might be interpreted as PMD due to their ambiguous features.                 |


### C: Categorize Errors

Out of 7 misclassified images, the errors can be grouped into three main categories. First, Scene/Context Confusion accounts for 2 images (approximately 28.6%). Next, Feature Similarity Issues are responsible for 4 images (about 57.1%). Lastly, Material Confusion is seen in 1 image (roughly 14.3%).

### D: Propose Improvements

Based on the quantitative analysis:

Hypotheses:

- Scene/Context Confusion (28.6%): The model may be distracted by cluttered backgrounds or contextual cues (e.g., a food court setting), leading it to misclassify objects.

- Feature Similarity Issues (57.1%): Many errors arise from subtle similarities in shape, texture, or color between classes, suggesting that the model's features are not sufficiently discriminative.

- Material Confusion (14.3%): The model struggles to distinguish fine differences in material properties (such as the neutral tones of cans versus cartons), resulting in misclassifications.

Potential Improvements:

- Enhance the Dataset: Increase diversity by including images with varied backgrounds, lighting conditions, and perspectives; add more examples for classes with subtle differences.

- Refine Data Augmentation: Use targeted augmentations (e.g., cropping or contrast adjustments) to highlight the object of interest and reduce background distractions.

- Improve Model Architecture: Incorporate attention mechanisms or fine-tune deeper layers to better capture discriminative features.

- Optimize Preprocessing: Consider higher resolution images or segmentation techniques to better isolate objects and capture subtle material details.